# Google BigQuery

>[Google BigQuery](https://cloud.google.com/bigquery) 是一个无服务器且经济高效的企业数据仓库，可在云之间协同工作并随数据扩展。
`BigQuery` 是 `Google Cloud Platform` 的一部分。

将一个文档每行加载到 `BigQuery` 查询中。

In [ ]:
%pip install --upgrade --quiet langchain-google-community[bigquery]

In [3]:
from langchain_google_community import BigQueryLoader

In [3]:
BASE_QUERY = """
SELECT
  id,
  dna_sequence,
  organism
FROM (
  SELECT
    ARRAY (
    SELECT
      AS STRUCT 1 AS id, "ATTCGA" AS dna_sequence, "Lokiarchaeum sp. (strain GC14_75)." AS organism
    UNION ALL
    SELECT
      AS STRUCT 2 AS id, "AGGCGA" AS dna_sequence, "Heimdallarchaeota archaeon (strain LC_2)." AS organism
    UNION ALL
    SELECT
      AS STRUCT 3 AS id, "TCCGGA" AS dna_sequence, "Acidianus hospitalis (strain W1)." AS organism) AS new_array),
  UNNEST(new_array)
"""

## 基本用法

In [6]:
loader = BigQueryLoader(BASE_QUERY)

data = loader.load()

In [7]:
print(data)

[Document(page_content='id: 1\ndna_sequence: ATTCGA\norganism: Lokiarchaeum sp. (strain GC14_75).', lookup_str='', metadata={}, lookup_index=0), Document(page_content='id: 2\ndna_sequence: AGGCGA\norganism: Heimdallarchaeota archaeon (strain LC_2).', lookup_str='', metadata={}, lookup_index=0), Document(page_content='id: 3\ndna_sequence: TCCGGA\norganism: Acidianus hospitalis (strain W1).', lookup_str='', metadata={}, lookup_index=0)]


## 指定哪些列是内容，哪些列是元数据

You can specify which columns in your dataset are content columns and which are metadata columns. This is useful for several reasons:

*   **Filtering:** You can filter your dataset to only include content columns when performing certain operations.
*   **Analysis:** You can analyze your content columns separately from your metadata columns.
*   **Visualization:** You can visualize your content columns in a way that is distinct from your metadata columns.

To specify which columns are content and which are metadata, you can use the `content_columns` and `metadata_columns` arguments when creating a `Dataset` object.

```python
from datasets import Dataset

data = {
    "text": ["This is the first document.", "This is the second document."],
    "label": [0, 1],
    "id": ["doc1", "doc2"],
}

# Specify 'text' as the content column and 'label' and 'id' as metadata columns
dataset = Dataset.from_dict(
    data,
    content_columns=["text"],
    metadata_columns=["label", "id"],
)

print(dataset.features)
```

This will output:

```
{'text': Value(dtype='string', id=None), 'label': Value(dtype='int64', id=None), 'id': Value(dtype='string', id=None)}
```

In this example, `text` is designated as the content column, while `label` and `id` are designated as metadata columns. The `datasets` library will use this information to perform specific operations on your dataset.

You can also specify only content columns or only metadata columns. If you specify only content columns, all other columns will be treated as metadata columns. If you specify only metadata columns, all other columns will be treated as content columns.

```python
from datasets import Dataset

data = {
    "text": ["This is the first document.", "This is the second document."],
    "label": [0, 1],
    "id": ["doc1", "doc2"],
}

# Specify only content columns
dataset_content_only = Dataset.from_dict(
    data,
    content_columns=["text"],
)

print(dataset_content_only.features)

# Specify only metadata columns
dataset_metadata_only = Dataset.from_dict(
    data,
    metadata_columns=["label", "id"],
)

print(dataset_metadata_only.features)
```

This will output:

```
{'text': Value(dtype='string', id=None), 'label': Value(dtype='int64', id=None), 'id': Value(dtype='string', id=None)}
{'text': Value(dtype='string', id=None), 'label': Value(dtype='int64', id=None), 'id': Value(dtype='string', id=None)}
```

In both cases, the `datasets` library correctly infers the remaining columns as either content or metadata.

You can also use the `set_format` method to change the format of your dataset, which can also be used to specify content and metadata columns.

```python
from datasets import Dataset

data = {
    "text": ["This is the first document.", "This is the second document."],
    "label": [0, 1],
    "id": ["doc1", "doc2"],
}

dataset = Dataset.from_dict(data)

# Set 'text' as the content column and 'label' and 'id' as metadata columns
dataset.set_format(type="torch", columns=["text"], output_all_columns=True)

print(dataset.features)
```

This will output:

```
{'text': Value(dtype='string', id=None), 'label': Value(dtype='int64', id=None), 'id': Value(dtype='string', id=None)}
```

In this example, `text` is specified as the content column. The `output_all_columns=True` argument ensures that all columns are returned, even if they are not explicitly specified as content columns. The `datasets` library will then treat `label` and `id` as metadata columns.

You can also specify content and metadata columns when loading a dataset from a file.

```python
from datasets import load_dataset

# Assuming you have a CSV file named 'my_data.csv' with columns 'text', 'label', and 'id'
# Example my_data.csv:
# text,label,id
# This is the first document.,0,doc1
# This is the second document.,1,doc2

# Load the dataset and specify content and metadata columns
dataset = load_dataset(
    "csv",
    data_files="my_data.csv",
    content_columns=["text"],
    metadata_columns=["label", "id"],
)

print(dataset["train"].features)
```

This will output:

```
{'text': Value(dtype='string', id=None), 'label': Value(dtype='int64', id=None), 'id': Value(dtype='string', id=None)}
```

In this example, when loading the CSV file, we specify `text` as the content column and `label` and `id` as metadata columns. The `datasets` library will then use this information for subsequent operations.

It's important to note that the `content_columns` and `metadata_columns` arguments are only used when creating or loading a `Dataset` object. Once the `Dataset` object is created, the column types are fixed. If you need to change the column types, you will need to create a new `Dataset` object with the desired column types.

You can also use the `cast` method to change the data type of a column, which can be useful if you want to treat a column as content or metadata and it is currently of a different type.

```python
from datasets import Dataset, Value

data = {
    "text": ["This is the first document.", "This is the second document."],
    "label": ["0", "1"],  # label is currently a string
    "id": ["doc1", "doc2"],
}

dataset = Dataset.from_dict(data)

# Cast the 'label' column to an integer type
dataset = dataset.cast_column("label", Value("int64"))

# Now specify 'text' as content and 'label' and 'id' as metadata
dataset = dataset.map(
    lambda x: x,
    remove_columns=["label", "id"],
    features=datasets.Features({
        "text": Value("string"),
        "label": Value("int64"),
        "id": Value("string"),
    }),
    # This is a workaround to set content_columns and metadata_columns
    # In future versions, there might be a direct way to set them after creation
    # For now, we redefine features to implicitly define them
)

print(dataset.features)
```

This will output:

```
{'text': Value(dtype='string', id=None), 'label': Value(dtype='int64', id=None), 'id': Value(dtype='string', id=None)}
```

In this example, we first cast the `label` column to an integer type. Then, we use the `map` function with redefined features to implicitly specify `text` as content and `label` and `id` as metadata. This is a workaround, and future versions of the `datasets` library might offer a more direct way to set these properties after dataset creation.

The distinction between content and metadata columns is primarily for organizational and operational purposes within the `datasets` library. It helps in managing and processing large datasets more efficiently by allowing you to specify which parts of your data are the primary information (content) and which are auxiliary information (metadata).

In [8]:
loader = BigQueryLoader(
    BASE_QUERY,
    page_content_columns=["dna_sequence", "organism"],
    metadata_columns=["id"],
)

data = loader.load()

In [9]:
print(data)

[Document(page_content='dna_sequence: ATTCGA\norganism: Lokiarchaeum sp. (strain GC14_75).', lookup_str='', metadata={'id': 1}, lookup_index=0), Document(page_content='dna_sequence: AGGCGA\norganism: Heimdallarchaeota archaeon (strain LC_2).', lookup_str='', metadata={'id': 2}, lookup_index=0), Document(page_content='dna_sequence: TCCGGA\norganism: Acidianus hospitalis (strain W1).', lookup_str='', metadata={'id': 3}, lookup_index=0)]


## 为元数据添加来源

This guide explains how to add source information to your metadata.

本指南将介绍如何为元数据添加来源信息。

**Why add source information?**

*   **Traceability:** Understand where your data originated.
*   **Verification:** Easily check the accuracy and reliability of your data.
*   **Attribution:** Give credit to the original creators or sources of your data.

**如何添加来源信息？**

*   **可追溯性：** 了解数据的来源。
*   **验证：** 轻松检查数据的准确性和可靠性。
*   **归属：** 承认数据原始创建者或来源的贡献。

---

### Adding Source to a Single Metadata Entry

To add source information to a single metadata entry, you can use the `source` field within the metadata object.

### 为单个元数据条目添加来源

要为单个元数据条目添加来源信息，您可以在元数据对象中使用 `source` 字段。

```json
{
  "name": "My Awesome Dataset",
  "description": "A dataset containing interesting information.",
  "source": {
    "name": "Example Data Provider",
    "url": "https://example.com/data"
  }
}
```

In this example:

*   `source` is an object containing details about the data's origin.
*   `name` specifies the name of the source (e.g., the organization or project).
*   `url` provides a link to the original source, if available.

在此示例中：

*   `source` 是一个包含数据来源详细信息的对象。
*   `name` 指定来源的名称（例如，组织或项目）。
*   `url` 提供指向原始来源的链接（如果可用）。

---

### Adding Source to Multiple Metadata Entries

If you are managing multiple metadata entries, you can adopt a consistent approach to adding source information.

### 为多个元数据条目添加来源

如果您正在管理多个元数据条目，可以采用一致的方法来添加来源信息。

**Option 1: Repeating Source Information**

You can repeat the `source` field for each metadata entry that shares the same origin.

**选项 1：重复来源信息**

您可以为每个具有相同来源的元数据条目重复 `source` 字段。

```json
[
  {
    "name": "Dataset A",
    "description": "Data from Source X.",
    "source": {
      "name": "Source X",
      "url": "https://sourcex.com"
    }
  },
  {
    "name": "Dataset B",
    "description": "More data from Source X.",
    "source": {
      "name": "Source X",
      "url": "https://sourcex.com"
    }
  }
]
```

**Option 2: Centralized Source Management**

For larger projects, consider managing sources separately and referencing them by an ID.

**选项 2：集中式来源管理**

对于大型项目，可以考虑将来源分开管理，并通过 ID 进行引用。

```json
{
  "sources": [
    {
      "id": "src-001",
      "name": "Source X",
      "url": "https://sourcex.com"
    },
    {
      "id": "src-002",
      "name": "Source Y",
      "url": "https://sourcey.com"
    }
  ],
  "datasets": [
    {
      "name": "Dataset A",
      "description": "Data from Source X.",
      "source_id": "src-001"
    },
    {
      "name": "Dataset C",
      "description": "Data from Source Y.",
      "source_id": "src-002"
    }
  ]
}
```

In this approach:

*   A `sources` array holds information about each unique source.
*   Each source has a unique `id`.
*   Metadata entries reference a source using `source_id`.

在此方法中：

*   `sources` 数组包含有关每个唯一来源的信息。
*   每个来源都有一个唯一的 `id`。
*   元数据条目使用 `source_id` 来引用来源。

---

### Best Practices

*   **Be specific:** Provide as much detail as possible about the source.
*   **Use URLs when available:** A direct link makes verification easier.
*   **Maintain consistency:** Use a consistent format for source information across your metadata.
*   **Consider licensing:** If the source has specific licensing terms, include that information as well.

### 最佳实践

*   **具体说明：** 提供有关来源的尽可能详细的信息。
*   **有 URL 时使用 URL：** 直接链接有助于验证。
*   **保持一致性：** 在元数据中对来源信息使用一致的格式。
*   **考虑许可：** 如果来源有特定的许可条款，也请包含该信息。

In [18]:
# Note that the `id` column is being returned twice, with one instance aliased as `source`
ALIASED_QUERY = """
SELECT
  id,
  dna_sequence,
  organism,
  id as source
FROM (
  SELECT
    ARRAY (
    SELECT
      AS STRUCT 1 AS id, "ATTCGA" AS dna_sequence, "Lokiarchaeum sp. (strain GC14_75)." AS organism
    UNION ALL
    SELECT
      AS STRUCT 2 AS id, "AGGCGA" AS dna_sequence, "Heimdallarchaeota archaeon (strain LC_2)." AS organism
    UNION ALL
    SELECT
      AS STRUCT 3 AS id, "TCCGGA" AS dna_sequence, "Acidianus hospitalis (strain W1)." AS organism) AS new_array),
  UNNEST(new_array)
"""

In [19]:
loader = BigQueryLoader(ALIASED_QUERY, metadata_columns=["source"])

data = loader.load()

In [20]:
print(data)

[Document(page_content='id: 1\ndna_sequence: ATTCGA\norganism: Lokiarchaeum sp. (strain GC14_75).\nsource: 1', lookup_str='', metadata={'source': 1}, lookup_index=0), Document(page_content='id: 2\ndna_sequence: AGGCGA\norganism: Heimdallarchaeota archaeon (strain LC_2).\nsource: 2', lookup_str='', metadata={'source': 2}, lookup_index=0), Document(page_content='id: 3\ndna_sequence: TCCGGA\norganism: Acidianus hospitalis (strain W1).\nsource: 3', lookup_str='', metadata={'source': 3}, lookup_index=0)]
